In [6]:
import json

In [ ]:
import pandas as pd

geo_lookup_matrix = {
    "Korean": ["5th gen k-pop", "classic k-pop", "classic korean pop", "joseon pop", "k-*", "korean*", "korea*"], 
    "Afghanistan": ["afghan pop"], 
    "African": ["african rock", "afrikaans*", "afro*", "cameroonian pop", "cape town indie", "ghanaian*", "kenyan*", "malawian*", "malian blues", "naija worship", "nigerian*", "south african*", "sudanese*", "tanzanian*"], 
    "Japanese": ["anime*", "j-*", "japanese*"], 
    "Arabic": ["arab*", "classic arab pop", "classic persian pop", "lebanese pop", "libyan pop", "egyptian*", "moroccan pop"], 
    "Argentine": ["argentine*"], 
    "Australian": ["aussie drill", "auckland indie", "australian*", "brisbane*", "melbourne*"], 
    "Austrian": ["austrian*", "austro-german modernism", "austropop"],
    "Indonesia": ["bali indie", "indonesian*"], 
    "Caribbean": ["barbadian pop", "jamaican*"],
    "Belgian": ["belgian*"], 
    "German": ["bergen indie", "frankfurt electronic", "german*", "neue deutsche harte", "neue neue deutsche welle"], 
    "UK": ["birmingham*", "brighton indie", "bristol*", "british*", "britpop", "britpop revival", "cambridgeshire indie", "cardiff indie", "celtic rock", "classic uk pop" ,"east anglia indie", "edmonton indie", "leeds indie", "leicester indie", "liverpool indie", "london*", "manchester*", "northern irish indie", "north east england indie", "nottingham indie", "uk *"], 
    "Scottish": ["scottish*"], 
    "Irish": ["dublin indie", "irish*"],
    "Portuguese": ["brazilian*", "portuguese*"], 
    "Baltic / Europe": ["bulgarian r&b", "czech pop", "eurodance", "europop", "euroska", "hungarian*", "moldovan pop"], 
    "Canadian": ["canadian*", "classic canadian rock"], 
    "Chinese": ["chinese*", "hong kong indie"], 
    "Scandinavian": ["classic danish pop", "danish*", "classic swedish pop", "finnish*", "icelandic*", "nordic*", "norwegian*", "scandinavian r&b", "scandipop", "swedish*", "viking metal"], 
    "Italian": ["classic italian pop", "italian*"], 
    "Spanish": ["colombian indie", "colombian pop", "dominican indie", "drill espanol", "el paso indie", "flamenco*", "guyanese pop", "latin*", "latintronica", "latinx alternative", "mexican*", "nuevo*", "pop venezolano", "ska mexicano", "spanish*", "trap latino"],         
    "Indian": ["desi*", "gujarati pop", "hindi indie", "indian*", "new delhi indie"], 
    "French": ["drill francais", "french*", "frenchcore"], 
    "Dutch": ["dutch*"], 
    "Ethiopian": ["ethio-jazz"],         
    "Greek": ["greek*"], 
    "Israeli": ["israeli*"], 
    "Malaysian": ["malaysian*"], 
    "Mongolian": ["mongolian alternative"], 
    "Polish": ["polish*"], 
    "Romanian": ["romanian*"], 
    "Russian": ["russian*"], 
    "Singaporean": ["singaporean"], 
    "Swiss": ["swiss*"], 
    "Taiwan": ["taiwan*"], 
    "Thai": ["thai*"], 
    "Turkish": ["turkish*"], 
    "Ukranian": ["ukranian*"], 
    "Vietnamese": ["viet*"], 
    "Welsh": ["welsh"], 
}

explicit_lookup = {}
wildcard_lookup = []

for region, patterns in geo_lookup_matrix.items():
    for pattern in patterns:
        p_clean = pattern.lower().strip()
        
        # If the string contains a literal wildcard character, chop it off and place it in prefixes
        if p_clean.endswith('*'):
            prefix_token = p_clean[:-1].strip()
            wildcard_lookup.append((prefix_token, region))
        else:
            explicit_lookup[p_clean] = region

wildcard_lookup.sort(key=lambda x: len(x[0]), reverse=True)


def find_region_for_genre(genre):
    genre = genre.lower().strip()

    # 1. Exact Match Check (O(1))
    if genre in explicit_lookup:
        return explicit_lookup[genre]
    
    # 2. Sequential Suffix Check
    for prefix, region in wildcard_lookup:
        if genre.startswith(prefix):
            return region
    
    return "Global"


def extract_geo_columns(genre_string):
    if pd.isna(genre_string):
        return [None, None, None]
    
    genre_str_clean = str(genre_string).strip()
    if not genre_str_clean:
        return [None, None, None]
    
    individual_genres = [g.strip() for g in genre_str_clean.split(",")]

    found_region = {}
    for g in individual_genres:
        if not g:
            continue
        region = find_region_for_genre(g)
        found_region[region] = found_region.get(region, 0) + 1

    # Sort items based on voting density, fallback to alphabetization on ties
    sorted_region = sorted(found_region.keys(), key=lambda r: (-found_region[r], r))

    # while len(sorted_region) < 3:
    #     sorted_region.append(None)

    return sorted_region[:3]

In [26]:
extract_geo_columns("k-pop, k-pop boy group")

['Korean', None, None]

In [35]:
df = pd.read_csv("spotify_artist_info.csv")[['names', 'genres']]

In [39]:
df[df['names'] == 'dreamcatcher']

,names,genres


In [41]:
df.head()
df['region'] = df['genres'].apply(lambda x: extract_geo_columns(x))
df['region'].value_counts()

region
[Global]                                 13505
[Korean]                                  2839
[Global, UK]                               549
[Scandinavian]                             297
[UK]                                       227
                                         ...  
[Global, Greek, Scandinavian]                1
[Global, Australian, Baltic / Europe]        1
[Global, Austrian]                           1
[Scandinavian, French]                       1
[Global, Scandinavian, UK]                   1
Name: count, Length: 139, dtype: int64

In [42]:
df

,names,genres,region
0,stayc,"k-pop, k-pop girl group",[Korean]
1,langhorne slim,"anti-folk, indie folk, modern folk rock, new a...",[Global]
2,luke bryan,"contemporary country, country, country road, m...",[Global]
3,the boyz,"k-pop, k-pop boy group",[Korean]
4,wendy,"k-pop, korean pop",[Korean]
...,...,...,...
20246,carole king,"brill building pop, classic rock, folk, folk r...",[Global]
20247,costa titch,south african trap,[African]
20248,camo,korean r&b,[Korean]
20249,maya randle,house,[Global]
